<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/wrongprecisiondataset/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Gerekli kütüphaneler
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from google.colab import drive
import matplotlib.pyplot as plt

# 2. Google Drive'ı bağla
drive.mount('/content/drive')

# 3. Model ve dataset yolları
model_path = "/content/drive/MyDrive/models/gp_model_24750"
dataset_path = "/content/drive/MyDrive/datasets/merged_europarl_informal.pkl"

# 4. Dataset'i yükle
with open(dataset_path, "rb") as f:
    df = pd.read_pickle(f)

# 5. Tokenizer ve model yükle (use_fast=False önemli!)
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 6. Dataset sınıfı
class GenderDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {key: val.squeeze(0) for key, val in encoding.items()}

# 7. Etiketleri encode et
label_map = {"female": 0, "male": 1}
df = df[df["gender"].isin(label_map)]
df["label"] = df["gender"].map(label_map)

# 8. Tüm veri için tahminleri yap
dataset = GenderDataset(df["text"].tolist(), tokenizer)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

model.eval()
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# 9. Tahminleri dataframe'e ekle
df["pred"] = all_preds
df["confidence"] = [max(p) for p in all_probs]

# 10. Yanlış tahmin edilenleri filtrele
wrong_df = df[df["label"] != df["pred"]].copy()

# 11. Sınıf dengesini sağla: confidence 0.6'dan başlayarak 0.9'a kadar sırayla sil
cls_counts = wrong_df["gender"].value_counts()
minority_class = cls_counts.idxmin()
majority_class = cls_counts.idxmax()
diff = cls_counts[majority_class] - cls_counts[minority_class]

filtered = wrong_df[
    (wrong_df["gender"] == majority_class) &
    (wrong_df["confidence"] > 0.6) &
    (wrong_df["confidence"] < 0.9)
].copy()

# confidence'e göre sırala
filtered = filtered.sort_values(by="confidence", ascending=True)

# sıradan silinecekleri seç
to_remove = filtered.iloc[:diff] if len(filtered) >= diff else filtered
balanced_df = wrong_df.drop(index=to_remove.index).reset_index(drop=True)

# 12. Sonuçları kontrol et
print("Son sınıf dağılımı:")
print(balanced_df["gender"].value_counts())

Mounted at /content/drive
Son sınıf dağılımı:
gender
female    164884
male      164884
Name: count, dtype: int64


In [ ]:
balanced_df

,text,gender,label,pred,confidence
0,To my chrome-worthy friend Vanessa-it was nice...,female,0,1,0.778225
1,How prepared are you to vaccinate animals whic...,female,0,1,0.536372
2,"Thank you, Commissioner and thank you also, Mr...",female,0,1,0.549011
3,It is essential that the work required continu...,female,0,1,0.555321
4,I had a job interview today. I also got a real...,female,0,1,0.539812
...,...,...,...,...,...
329763,"Since then, we have been trying to resolve thi...",male,1,0,0.585658
329764,I'm so sick of people telling me that I can't ...,male,1,0,0.711817
329765,No you know what? I don't hate myself. I fucki...,male,1,0,0.537574
329766,I would like to point out that apart from the ...,male,1,0,0.728995


In [ ]:
# 13. İstersen kaydet
balanced_df_path = "/content/drive/MyDrive/datasets/balanced_wrong_predictions.pkl"
with open(balanced_df_path, "wb") as f:
    pd.to_pickle(balanced_df, f)
print(f"\nBalanced dataframe kaydedildi: {balanced_df_path}")


Balanced dataframe kaydedildi: /content/drive/MyDrive/datasets/balanced_wrong_predictions.pkl


In [ ]:
import requests

def send_telegram_message(message):
    token = "7791020893:AAGIXZbLRG6YVNXaNhRkCNiQDUV2-jXsDJY"
    chat_id = 7689600055
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    data = {"chat_id": chat_id, "text": message}
    requests.post(url, data=data)

send_telegram_message("Colab çalışman tamamlandı 🎉 Yanlış Tahminlerden Oluşan dataset oluşturuldu")